# 02 — Classification et comparaison

Baseline entraînée sur les textes train uniquement : TF-IDF uni/bigrammes + régression logistique pondérée. Comparaison avec une prédiction de classe majoritaire, sans ajuster de paramètres sur la validation. DistilBERT reste une extension.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Ouvrir depuis la racine ou notebooks/'
sys.path.insert(0, str(ROOT))
import json
import sqlite3
from contextlib import closing
import pandas as pd
REPORTS = ROOT / 'reports/demo'
DATABASE = ROOT / 'data/processed/demo.sqlite'
assert DATABASE.is_file(), 'Executer python -m scripts.prepare_demo depuis la racine'


In [2]:
import joblib
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, classification_report
model = joblib.load(ROOT / 'artifacts/demo/baseline.joblib')
with closing(sqlite3.connect(DATABASE)) as db:
    train = pd.read_sql_query("SELECT narrative, product FROM complaints WHERE split='train' ORDER BY text_hash LIMIT 30000", db)
    validation = pd.read_sql_query("SELECT narrative, product FROM complaints WHERE split='validation' ORDER BY text_hash LIMIT 10000", db)
labels = sorted(set(train['product']) | set(validation['product']))
dummy = DummyClassifier(strategy='most_frequent').fit(train[['narrative']], train['product'])
predictions = model.predict(validation['narrative'])
comparison = pd.DataFrame([
    {'modele': 'Classe majoritaire', 'macro_f1': f1_score(validation['product'], dummy.predict(validation[['narrative']]), labels=labels, average='macro', zero_division=0)},
    {'modele': 'TF-IDF + regression logistique', 'macro_f1': f1_score(validation['product'], predictions, labels=labels, average='macro', zero_division=0)}
])
comparison

,modele,macro_f1
0,Classe majoritaire,0.073338
1,TF-IDF + regression logistique,0.496604


In [3]:
pd.DataFrame(classification_report(validation['product'], predictions, labels=labels, output_dict=True, zero_division=0)).T

,precision,recall,f1-score,support
Checking or savings account,0.571429,0.923077,0.705882,26.000000
Credit card,0.551724,0.800000,0.653061,20.000000
Credit reporting or other personal consumer reports,0.929936,0.869048,0.898462,336.000000
Debt collection,0.525641,0.661290,0.585714,62.000000
Debt or credit management,1.000000,0.333333,0.500000,3.000000
"Money transfer, virtual currency, or money service",0.800000,0.470588,0.592593,17.000000
Mortgage,0.857143,0.600000,0.705882,10.000000
"Payday loan, title loan, personal loan, or advance loan",0.000000,0.000000,0.000000,6.000000
Prepaid card,0.000000,0.000000,0.000000,4.000000
Student loan,0.333333,0.571429,0.421053,7.000000


## Analyse des erreurs
Les cinq premières erreurs dans l’ordre stable des hash sont affichées sans publier les textes. Une classe sans effectif de validation ne permet pas d’estimer sa performance. La macro-F1 donne le même poids à chaque classe.

In [4]:
errors = validation.assign(prediction=predictions)
errors.loc[errors['product'] != errors['prediction'], ['product', 'prediction']].head()

,product,prediction
3,Credit reporting or other personal consumer re...,Debt collection
6,"Money transfer, virtual currency, or money ser...",Checking or savings account
12,Credit reporting or other personal consumer re...,Debt collection
18,Debt collection,Credit reporting or other personal consumer re...
33,Debt or credit management,Credit reporting or other personal consumer re...


Le modèle Joblib doit provenir d’une source de confiance. Le test temporel 2026 reste fermé pendant ces expériences. La reproductibilité dépend des versions consignées dans manifest.json.